# Heston FNO — Results, Evaluation & Pricing Surface

## 1. Colab setup — use the same paths as `colab_train(1).ipynb`

This notebook intentionally uses the exact Colab repository location and Drive/data layout from your training notebook.

In [ ]:
# ============================================================
# STEP 1 — Clone repo + install project
# Same setup as colab_train(1).ipynb
# ============================================================

import os
import sys
import shutil
import math
import time
from pathlib import Path

import torch

REPO = Path("/content/FNO-for-Heston-Pricing-Engine-")
ROOT = REPO

# ============================================================
# SHARED RESULTS SETTINGS — define these before any later cell
# ============================================================

EVAL_MAX_SAMPLES = None
SURFACE_TEST_POSITION = 0

FNO_BENCHMARK_BATCH_SIZE = 4
FNO_WARMUP = 5
FNO_REPEATS = 20

RUN_ADI_BENCHMARK = False
ADI_STRIKE = 100.0
ADI_RATE = 0.05
ADI_MATURITY = 1.0
ADI_N_STEPS = 100
ADI_WARMUP = 1
ADI_REPEATS = 3

MC_PATHS = 5000
MC_STEPS = 100
MC_SEED = 42
MC_DEVICE = "cpu"

RESULTS_DIR = REPO / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not REPO.exists():
    !git clone https://github.com/bananapants-87/FNO-for-Heston-Pricing-Engine-.git
else:
    # The Colab folder may already exist from an older notebook session.
    # Refresh tracked project files so Python does not use stale source code.
    !git -C /content/FNO-for-Heston-Pricing-Engine- fetch origin main -q
    !git -C /content/FNO-for-Heston-Pricing-Engine- reset --hard origin/main -q

os.chdir(REPO)

# Same install approach as the training notebook.
!pip install -q neuraloperator pyyaml
!pip install -q -e . --no-deps

print("Working directory:", os.getcwd())
print("Repository exists:", REPO.exists())
print("Results settings initialized successfully.")

# ============================================================
# STEP 2 — Mount Drive and copy the same four data files
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = REPO / "data" / "final"
DATA_DIR.mkdir(parents=True, exist_ok=True)

files = ["inputs.pt", "targets.pt", "s_grid.pt", "v_grid.pt"]

candidates = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/MyDrive/final"),
    Path("/content/final"),
    Path("/content"),
]

for fname in files:
    destination = DATA_DIR / fname

    if destination.exists():
        print(fname, "already present")
        continue

    found = False
    for location in candidates:
        source = location / fname
        if source.exists():
            shutil.copy2(source, destination)
            print("Copied:", fname, "from", source)
            found = True
            break

    if not found:
        print("MISSING:", fname)

print("\n--- DATA VERIFICATION ---")
for fname in files:
    p = DATA_DIR / fname
    print(f"{fname:12} ->", "FOUND" if p.exists() else "MISSING")

# ============================================================
# STEP 3 — Find the trained checkpoint
# ============================================================

CHECKPOINT_PATH = REPO / "checkpoints" / "heston_fno.pth"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Same-runtime case: training already created the checkpoint here.
# Restarted-runtime case: try common Drive locations.
drive_checkpoint_candidates = [
    Path("/content/drive/MyDrive/FNO-Heston-Project/run_002_5000train_1000val_10epochs/heston_fno.pth"),
    Path("/content/drive/MyDrive/heston_fno.pth"),
    Path("/content/drive/MyDrive/checkpoints/heston_fno.pth"),
    Path("/content/drive/MyDrive/FNO-for-Heston-Pricing-Engine-/checkpoints/heston_fno.pth"),
]

if not CHECKPOINT_PATH.exists():
    for source in drive_checkpoint_candidates:
        if source.exists():
            shutil.copy2(source, CHECKPOINT_PATH)
            print("Recovered checkpoint from:", source)
            break

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "\nTrained checkpoint not found.\n"
        f"Expected: {CHECKPOINT_PATH}\n\n"
        "Run STEP 6 (`!python scripts/train_fno.py`) in your training notebook "
        "and then run this results notebook in the SAME Colab runtime.\n\n"
        "For future sessions, save the checkpoint to Drive after training with:\n"
        "shutil.copy2('/content/FNO-for-Heston-Pricing-Engine-/checkpoints/heston_fno.pth', "
        "'/content/drive/MyDrive/heston_fno.pth')"
    )

print("\nCheckpoint:", CHECKPOINT_PATH)
print("Checkpoint size (MB):", round(CHECKPOINT_PATH.stat().st_size / 1024**2, 2))

In [ ]:
# ============================================================
# IMPORT PROJECT MODULES
# Make the repository src/ directory available before importing heston_fno.
# This makes the notebook robust even when the editable pip install is not
# picked up by the current Jupyter kernel.
# ============================================================

import sys
from pathlib import Path

REPO = Path("/content/FNO-for-Heston-Pricing-Engine-")
ROOT = REPO
SRC = REPO / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

if not SRC.exists():
    raise FileNotFoundError(f"Project source directory not found: {SRC}")

# Remove any heston_fno modules loaded from an older checkout in this kernel.
for module_name in list(sys.modules):
    if module_name == "heston_fno" or module_name.startswith("heston_fno."):
        del sys.modules[module_name]

import torch
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

from heston_fno.data.dataset import HestonDataset
from heston_fno.data.preprocessing import split_indices
from heston_fno.data.normalization import StandardScaler
from heston_fno.models.fno import build_model_from_config
from heston_fno.evaluation.errors import summarize_errors
from heston_fno.evaluation.greeks import (
    compute_delta,
    compute_vega,
    compute_reference_delta,
    compare_greeks,
)
from heston_fno.benchmarking.fno import benchmark_fno
from heston_fno.benchmarking.adi import solve_heston_adi
from heston_fno.benchmarking.monte_carlo import price_heston_call_mc_curve

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Working directory:", Path.cwd())
print("Project source:", SRC)
print("PyTorch:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Load the trained checkpoint, saved configuration and scalers

The checkpoint is the file produced by `scripts/train_fno.py` at:

`/content/FNO-for-Heston-Pricing-Engine-/checkpoints/heston_fno.pth`

The training script stores the model state, configuration, training history, input scaler and target scaler in this file.

In [3]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

saved_config = checkpoint.get("config", None)
if saved_config is None:
    with open(ROOT / "configs" / "fno.yaml", "r", encoding="utf-8") as f:
        saved_config = yaml.safe_load(f) or {}

model = build_model_from_config(saved_config).to(device)

state_dict = checkpoint["model_state_dict"]
state_dict = {k: v for k, v in state_dict.items() if k != "_metadata"}
model.load_state_dict(state_dict)
model.eval()

input_scaler = StandardScaler()
target_scaler = StandardScaler()

if "input_scaler" in checkpoint and "target_scaler" in checkpoint:
    input_scaler.load_state_dict(checkpoint["input_scaler"])
    target_scaler.load_state_dict(checkpoint["target_scaler"])
else:
    raise KeyError(
        "Checkpoint does not contain input_scaler/target_scaler. "
        "Use a checkpoint created by the current train_fno.py."
    )

history = checkpoint.get("history", {})

print("Checkpoint loaded successfully.")
print("Saved epoch:", checkpoint.get("epoch", "not stored"))
print("Best validation loss:", checkpoint.get("best_val_loss", "not stored"))
print("Saved model config:", saved_config.get("model", {}))
print("History keys:", list(history.keys()))

Checkpoint loaded successfully.
Saved epoch: 10
Best validation loss: 1.6694291324321447e-05
Saved model config: {'n_modes': [8, 8], 'hidden_channels': 32, 'in_channels': 7, 'out_channels': 1}
History keys: ['train_loss', 'val_loss']


## 3. Load the complete dataset and recreate the exact deterministic split

The repo uses an 80/10/10 split with seed `42` in the current configuration.

We use the raw dataset here, then apply the scaler stored in the trained checkpoint.
This avoids accidentally fitting new normalization statistics during evaluation.

In [4]:
full_dataset = HestonDataset(
    inputs_path=DATA_DIR / "inputs.pt",
    targets_path=DATA_DIR / "targets.pt",
    s_grid_path=DATA_DIR / "s_grid.pt",
    v_grid_path=DATA_DIR / "v_grid.pt",
)

split_cfg = saved_config.get("training", {}).get("split", [0.8, 0.1, 0.1])
seed = saved_config.get("training", {}).get("seed", 42)

train_idx, val_idx, test_idx = split_indices(
    len(full_dataset),
    split=split_cfg,
    seed=seed,
)

print("Total samples:", len(full_dataset))
print("Train samples:", len(train_idx))
print("Validation samples:", len(val_idx))
print("Test samples:", len(test_idx))
print("Input sample shape:", tuple(full_dataset[0][0].shape))
print("Target sample shape:", tuple(full_dataset[0][1].shape))
print("S grid shape:", tuple(full_dataset.s_grid.shape))
print("v grid shape:", tuple(full_dataset.v_grid.shape))

Total samples: 50000
Train samples: 40000
Validation samples: 5000
Test samples: 5000
Input sample shape: (7, 64, 32)
Target sample shape: (1, 64, 32)
S grid shape: (64,)
v grid shape: (32,)


In [ ]:
# ============================================================
# RESULTS-NOTEBOOK SETTINGS VERIFICATION
# ============================================================

required_settings = {
    "EVAL_MAX_SAMPLES": EVAL_MAX_SAMPLES,
    "SURFACE_TEST_POSITION": SURFACE_TEST_POSITION,
    "FNO_BENCHMARK_BATCH_SIZE": FNO_BENCHMARK_BATCH_SIZE,
    "FNO_WARMUP": FNO_WARMUP,
    "FNO_REPEATS": FNO_REPEATS,
    "RUN_ADI_BENCHMARK": RUN_ADI_BENCHMARK,
    "ADI_STRIKE": ADI_STRIKE,
    "ADI_RATE": ADI_RATE,
    "ADI_MATURITY": ADI_MATURITY,
    "ADI_N_STEPS": ADI_N_STEPS,
    "MC_PATHS": MC_PATHS,
    "MC_STEPS": MC_STEPS,
    "MC_SEED": MC_SEED,
    "MC_DEVICE": MC_DEVICE,
}

print("Results settings verified.")
for key, value in required_settings.items():
    print(f"  {key} = {value}")

In [6]:
class ScaledIndexDataset(Dataset):
    def __init__(self, base_dataset, indices, input_scaler, target_scaler):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.input_scaler = input_scaler
        self.target_scaler = target_scaler

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        x_raw, y_raw = self.base_dataset[idx]
        x = self.input_scaler.transform(x_raw)
        y = self.target_scaler.transform(y_raw)
        return x, y

test_indices_used = list(test_idx)
if EVAL_MAX_SAMPLES is not None:
    test_indices_used = test_indices_used[: int(EVAL_MAX_SAMPLES)]

test_scaled_dataset = ScaledIndexDataset(
    full_dataset,
    test_indices_used,
    input_scaler,
    target_scaler,
)

eval_batch_size = int(
    saved_config.get("training", {}).get("batch_size", 4)
)

test_loader = DataLoader(
    test_scaled_dataset,
    batch_size=eval_batch_size,
    shuffle=False,
    num_workers=0,
)

print("Samples used for evaluation:", len(test_scaled_dataset))
print("Evaluation batch size:", eval_batch_size)

Samples used for evaluation: 5000
Evaluation batch size: 16


## 4. Evaluate the held-out test set

These metrics are computed in **physical option-price units** after inverse normalization.

The per-point mean relative error is retained for consistency with `scripts/evaluate.py`,
but it can become very large near target prices close to zero. The relative L2 error is
also reported because it is less sensitive to individual near-zero grid points.

In [7]:
model.eval()

sum_sq = 0.0
sum_abs = 0.0
sum_rel = 0.0
sum_y = 0.0
sum_y2 = 0.0
sum_count = 0
max_abs = 0.0

with torch.no_grad():
    for x_scaled, y_scaled in test_loader:
        x_scaled = x_scaled.to(device)
        y_scaled = y_scaled.to(device)

        pred_scaled = model(x_scaled)

        pred = target_scaler.inverse_transform(pred_scaled)
        y = target_scaler.inverse_transform(y_scaled)

        diff = pred - y
        abs_diff = torch.abs(diff)

        sum_sq += torch.sum(diff ** 2).item()
        sum_abs += torch.sum(abs_diff).item()
        sum_rel += torch.sum(abs_diff / (torch.abs(y) + 1e-6)).item()

        sum_y += torch.sum(y).item()
        sum_y2 += torch.sum(y ** 2).item()
        sum_count += y.numel()

        batch_max = abs_diff.max().item()
        max_abs = max(max_abs, batch_max)

rmse_value = math.sqrt(sum_sq / max(sum_count, 1))
mae_value = sum_abs / max(sum_count, 1)
mre_value = sum_rel / max(sum_count, 1)

relative_l2_value = math.sqrt(sum_sq) / (math.sqrt(sum_y2) + 1e-7)

# Global R² over all price-grid points.
mean_y = sum_y / max(sum_count, 1)
sst = sum_y2 - 2.0 * mean_y * sum_y + sum_count * (mean_y ** 2)
r2_value = 1.0 - (sum_sq / sst) if sst > 0 else float("nan")

metrics = {
    "RMSE": rmse_value,
    "MAE": mae_value,
    "Maximum Absolute Error": max_abs,
    "Mean Relative Error": mre_value,
    "Relative L2 Error": relative_l2_value,
    "R2": r2_value,
}

metrics_df = pd.DataFrame(
    [{"Metric": k, "Value": v} for k, v in metrics.items()]
)

display(metrics_df)

metrics_df.to_csv(RESULTS_DIR / "evaluation_metrics.csv", index=False)

print("\nMetrics saved to:", RESULTS_DIR / "evaluation_metrics.csv")

NameError: name 'math' is not defined

## 5. Training and validation loss curve

In [ ]:
train_loss = history.get("train_loss", [])
val_loss = history.get("val_loss", [])

if len(train_loss) == 0:
    print("No training history found in the checkpoint.")
else:
    epochs = np.arange(1, len(train_loss) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_loss, marker="o", label="Training loss")
    if len(val_loss) == len(train_loss):
        plt.plot(epochs, val_loss, marker="o", label="Validation loss")

    plt.xlabel("Epoch")
    plt.ylabel("MSE loss (normalized target)")
    plt.title("FNO Training and Validation Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "training_validation_loss.png", dpi=200)
    plt.show()

## 6. Select one held-out test sample and predict its full pricing surface

In [ ]:
if SURFACE_TEST_POSITION < 0 or SURFACE_TEST_POSITION >= len(test_idx):
    raise IndexError(
        f"SURFACE_TEST_POSITION must be between 0 and {len(test_idx)-1}."
    )

surface_raw_index = int(test_idx[SURFACE_TEST_POSITION])
params = full_dataset.params[surface_raw_index]

x_raw, y_raw = full_dataset[surface_raw_index]

x_scaled = input_scaler.transform(x_raw).unsqueeze(0)
y_scaled = target_scaler.transform(y_raw).unsqueeze(0)

with torch.no_grad():
    pred_scaled = model(x_scaled.to(device))

pred_surface = target_scaler.inverse_transform(pred_scaled).cpu()[0]
true_surface = y_raw.unsqueeze(0).cpu()[0]

abs_error_surface = torch.abs(pred_surface - true_surface)

s_grid = full_dataset.s_grid.cpu()
v_grid = full_dataset.v_grid.cpu()

S, V = torch.meshgrid(s_grid, v_grid, indexing="ij")
S_np = S.numpy()
V_np = V.numpy()

pred_np = pred_surface.squeeze(0).numpy()
true_np = true_surface.squeeze(0).numpy()
err_np = abs_error_surface.squeeze(0).numpy()

print("Original dataset sample index:", surface_raw_index)
print("Parameter values [kappa, theta, sigma, rho, v0]:")
print(full_dataset.params[surface_raw_index].numpy())
print("Surface shape:", pred_surface.shape)
print("True surface min/max:", true_surface.min().item(), true_surface.max().item())
print("Predicted surface min/max:", pred_surface.min().item(), pred_surface.max().item())

## 7. Ground-truth pricing surface

This is the reference pricing surface stored in `targets.pt`.
In your project these are the prices generated by the reference PDE/ADI workflow used to
construct the dataset.

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_np, V_np, true_np, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel("Spot price S")
ax.set_ylabel("Variance v")
ax.set_zlabel("Option price")
ax.set_title("Ground-Truth Heston Pricing Surface")
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.1, label="Price")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "ground_truth_pricing_surface_3d.png", dpi=200)
plt.show()

## 8. FNO predicted pricing surface

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_np, V_np, pred_np, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel("Spot price S")
ax.set_ylabel("Variance v")
ax.set_zlabel("Option price")
ax.set_title("FNO Predicted Heston Pricing Surface")
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.1, label="Price")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fno_predicted_pricing_surface_3d.png", dpi=200)
plt.show()

## 9. Absolute-error surface

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_np, V_np, err_np, cmap="magma", linewidth=0, antialiased=True)
ax.set_xlabel("Spot price S")
ax.set_ylabel("Variance v")
ax.set_zlabel("Absolute error")
ax.set_title("|FNO prediction − Ground truth|")
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.1, label="Absolute error")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "absolute_error_surface_3d.png", dpi=200)
plt.show()

## 10. 2D heatmaps

The 2D views are often easier to put into a report than 3D plots.

In [ ]:
fig = plt.figure(figsize=(8, 5.5))
plt.imshow(
    true_np.T,
    origin="lower",
    aspect="auto",
    extent=[s_grid.min().item(), s_grid.max().item(), v_grid.min().item(), v_grid.max().item()],
)
plt.xlabel("Spot price S")
plt.ylabel("Variance v")
plt.title("Ground-Truth Pricing Surface")
plt.colorbar(label="Option price")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ground_truth_pricing_surface_2d.png", dpi=200)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 5.5))
plt.imshow(
    pred_np.T,
    origin="lower",
    aspect="auto",
    extent=[s_grid.min().item(), s_grid.max().item(), v_grid.min().item(), v_grid.max().item()],
)
plt.xlabel("Spot price S")
plt.ylabel("Variance v")
plt.title("FNO Predicted Pricing Surface")
plt.colorbar(label="Option price")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "fno_predicted_pricing_surface_2d.png", dpi=200)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 5.5))
plt.imshow(
    err_np.T,
    origin="lower",
    aspect="auto",
    extent=[s_grid.min().item(), s_grid.max().item(), v_grid.min().item(), v_grid.max().item()],
)
plt.xlabel("Spot price S")
plt.ylabel("Variance v")
plt.title("Absolute Pricing Error")
plt.colorbar(label="Absolute error")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "absolute_error_surface_2d.png", dpi=200)
plt.show()

## 11. Predicted-vs-true price comparison

This checks how closely individual grid-point predictions follow the reference prices.

In [ ]:
true_flat = true_np.ravel()
pred_flat = pred_np.ravel()

fig = plt.figure(figsize=(7, 6))
plt.scatter(true_flat, pred_flat, s=8, alpha=0.35)

lo = min(true_flat.min(), pred_flat.min())
hi = max(true_flat.max(), pred_flat.max())
plt.plot([lo, hi], [lo, hi], linestyle="--", label="Perfect prediction")

plt.xlabel("Ground-truth option price")
plt.ylabel("FNO predicted option price")
plt.title("FNO Prediction vs Ground Truth")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "prediction_vs_ground_truth.png", dpi=200)
plt.show()

## 12. Absolute-error histogram

In [ ]:
fig = plt.figure(figsize=(8, 5))
plt.hist(np.abs(pred_flat - true_flat), bins=50)
plt.xlabel("Absolute error")
plt.ylabel("Number of grid points")
plt.title("Distribution of Absolute Pricing Error")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "absolute_error_histogram.png", dpi=200)
plt.show()

## 13. Pricing cross-section at one variance level

This gives a conventional 2D pricing curve:

`S → option price`

for a fixed variance-grid index.

In [ ]:
v_index = len(v_grid) // 2
v_value = v_grid[v_index].item()

fig = plt.figure(figsize=(8, 5))
plt.plot(s_grid.numpy(), true_np[:, v_index], marker="o", markersize=3, label="Ground truth")
plt.plot(s_grid.numpy(), pred_np[:, v_index], marker="x", markersize=3, label="FNO")

plt.xlabel("Spot price S")
plt.ylabel("Option price")
plt.title(f"Pricing Cross-Section at variance v = {v_value:.6g}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "pricing_cross_section.png", dpi=200)
plt.show()

## 14. Error summary for the selected surface

In [ ]:
surface_summary = summarize_errors(
    pred_surface.unsqueeze(0),
    true_surface.unsqueeze(0),
).to_dict()

surface_summary_df = pd.DataFrame(
    [{"Metric": k, "Value": v} for k, v in surface_summary.items()]
)
display(surface_summary_df)
surface_summary_df.to_csv(RESULTS_DIR / "selected_surface_metrics.csv", index=False)

## 15. Greeks / sensitivities

The repository provides:

- `compute_delta()` — derivative of the FNO output with respect to the spot-price `S` input channel.
- `compute_vega()` — derivative with respect to the initial variance `v0` input channel, so this is best
  described here as **Vega-like / v0 sensitivity**.

A reference Delta can be obtained directly from the ground-truth pricing surface by numerical differentiation.

The repository's `compute_reference_vega()` differentiates the pricing surface with respect to the
grid coordinate `v`; that is **not the same derivative** as `dV/dv0`, so this notebook does not
present those two quantities as a direct Vega-vs-reference comparison.

In [ ]:
# FNO Delta and v0-sensitivity for the selected sample.
delta_surface = compute_delta(
    model=model,
    x=x_scaled,
    input_scaler=input_scaler,
    target_scaler=target_scaler,
    s_channel=5,
    device=device,
)[0]

vega_like_surface = compute_vega(
    model=model,
    x=x_scaled,
    input_scaler=input_scaler,
    target_scaler=target_scaler,
    v0_channel=4,
    device=device,
)[0]

reference_delta = compute_reference_delta(
    true_surface,
    s_grid,
)[0]

delta_metrics = compare_greeks(
    delta_surface,
    reference_delta,
)

print("Delta comparison metrics:")
display(pd.DataFrame([{"Metric": k, "Value": v} for k, v in delta_metrics.items()]))

pd.DataFrame(
    [{"Metric": k, "Value": v} for k, v in delta_metrics.items()]
).to_csv(RESULTS_DIR / "delta_comparison_metrics.csv", index=False)

### Delta surface

In [ ]:
delta_np = delta_surface.squeeze(0).numpy()

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_np, V_np, delta_np, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel("Spot price S")
ax.set_ylabel("Variance v")
ax.set_zlabel("Delta-like sensitivity")
ax.set_title("FNO Delta-like Sensitivity Surface")
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.1, label="Delta")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fno_delta_surface_3d.png", dpi=200)
plt.show()

### Vega-like / v0-sensitivity surface

In [ ]:
vega_like_np = vega_like_surface.squeeze(0).numpy()

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_np, V_np, vega_like_np, cmap="plasma", linewidth=0, antialiased=True)
ax.set_xlabel("Spot price S")
ax.set_ylabel("Variance v")
ax.set_zlabel("dPrice/dv0")
ax.set_title("FNO Vega-like (Initial Variance Sensitivity) Surface")
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.1, label="dPrice/dv0")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fno_vega_like_surface_3d.png", dpi=200)
plt.show()

## 16. FNO inference-time benchmark

This uses the repository's timing helper with warm-up runs and repeated inference.

In [ ]:
bench_n = min(FNO_BENCHMARK_BATCH_SIZE, len(test_scaled_dataset))
bench_x = torch.stack([test_scaled_dataset[i][0] for i in range(bench_n)], dim=0)

fno_benchmark_result = benchmark_fno(
    model=model,
    batch=bench_x,
    device=device,
    warmup=FNO_WARMUP,
    repeats=FNO_REPEATS,
)

fno_time = fno_benchmark_result.average_time
fno_throughput = fno_benchmark_result.throughput

fno_results_df = pd.DataFrame([
    {"Metric": "FNO average inference time (s/batch)", "Value": fno_time},
    {"Metric": "FNO throughput (samples/s)", "Value": fno_throughput},
])

display(fno_results_df)
fno_results_df.to_csv(RESULTS_DIR / "fno_benchmark.csv", index=False)

## 17. Optional ADI benchmark

This cell is **disabled by default**.

The repository contains an HV-ADI reference solver and timing helper. The benchmark parameters
(`strike`, `rate`, `maturity`, and `n_steps`) should match the reference setup you intend to report.

The ADI helper expects the first five channels to contain the **raw Heston parameters**, so the
benchmark below deliberately uses raw inputs rather than normalized FNO inputs.

Run this only after the FNO results are working.

In [ ]:
if RUN_ADI_BENCHMARK:
    from heston_fno.benchmarking.adi import (
        create_adi_benchmark_fn,
        benchmark_adi,
    )

    # Raw input tensors: first 5 channels are raw Heston parameters.
    raw_batch = torch.stack(
        [full_dataset[int(test_idx[i])][0] for i in range(bench_n)],
        dim=0,
    )

    adi_fn = create_adi_benchmark_fn(
        s_grid=s_grid,
        v_grid=v_grid,
        strike=ADI_STRIKE,
        r=ADI_RATE,
        maturity=ADI_MATURITY,
        n_steps=ADI_N_STEPS,
    )

    adi_result = benchmark_adi(
        adi_fn=adi_fn,
        batch=raw_batch,
        warmup=ADI_WARMUP,
        repeats=ADI_REPEATS,
    )

    adi_time = adi_result.average_time
    adi_throughput = adi_result.throughput

    comparison = pd.DataFrame([
        {"Method": "FNO", "Average time (s/batch)": fno_time, "Throughput (samples/s)": fno_throughput},
        {"Method": "ADI", "Average time (s/batch)": adi_time, "Throughput (samples/s)": adi_throughput},
    ])

    comparison["Speedup vs ADI"] = np.nan
    comparison.loc[comparison["Method"] == "FNO", "Speedup vs ADI"] = adi_time / fno_time

    display(comparison)
    comparison.to_csv(RESULTS_DIR / "fno_vs_adi_benchmark.csv", index=False)

else:
    print("ADI benchmark skipped. Set RUN_ADI_BENCHMARK = True to run it.")

In [ ]:
# ============================================================
# OPTIONAL — Save checkpoint to Google Drive for future sessions
# ============================================================

SAVE_CHECKPOINT_TO_DRIVE = False

if SAVE_CHECKPOINT_TO_DRIVE:
    drive_destination = Path("/content/drive/MyDrive/heston_fno.pth")
    shutil.copy2(CHECKPOINT_PATH, drive_destination)
    print("Checkpoint saved to:", drive_destination)
else:
    print("Checkpoint not copied to Drive. Set SAVE_CHECKPOINT_TO_DRIVE = True when desired.")

In [ ]:
# ============================================================
# GENERATE COMPLETE 64 × 32 ADI SURFACE FOR COMPARISON
# ============================================================

adi_surface_start = time.perf_counter()

adi_surface = solve_heston_adi(
    params=params,
    s_grid=s_grid,
    v_grid=v_grid,
    strike=ADI_STRIKE,
    r=ADI_RATE,
    maturity=ADI_MATURITY,
    n_steps=ADI_N_STEPS,
)

adi_surface_generation_time = time.perf_counter() - adi_surface_start

assert tuple(adi_surface.shape) == (64, 32)

print("ADI surface shape:", tuple(adi_surface.shape))
print(f"ADI complete-surface generation time: {adi_surface_generation_time:.6f} s")


## 18. Monte Carlo — complete 64 × 32 pricing surface

This section adds Monte Carlo to the existing results notebook without removing or replacing the earlier FNO evaluation, pricing surfaces, Greeks, FNO benchmark, ADI benchmark, or report summary.

The existing Monte Carlo implementation prices all 64 spot values for one initial variance. It is therefore evaluated for each of the 32 variance-grid values to construct the complete 64 × 32 surface.


In [ ]:
# ============================================================
# 18A. MONTE CARLO IMPORTS AND SETTINGS
# ============================================================

from heston_fno.benchmarking.adi import solve_heston_adi
from heston_fno.benchmarking.monte_carlo import price_heston_call_mc_curve

MC_PATHS = 5000
MC_STEPS = 100
MC_SEED = 42

# Keep these identical to the pricing setup used for ADI.
MC_STRIKE = 100.0
MC_RATE = 0.0
MC_MATURITY = 1.0

# The MC implementation supports both CPU and CUDA.
mc_device = torch.device(MC_DEVICE)

print('Monte Carlo device:', mc_device)
print('Paths per variance slice:', MC_PATHS)
print('Euler time steps:', MC_STEPS)
print('Target surface:', (len(s_grid), len(v_grid)))


In [ ]:
# ============================================================
# 18B. GENERATE THE COMPLETE 64 × 32 MONTE CARLO SURFACE
# ============================================================

mc_price_columns = []
mc_se_columns = []
mc_low_columns = []
mc_high_columns = []

mc_surface_start = time.perf_counter()

for j, v0 in enumerate(v_grid):
    result = price_heston_call_mc_curve(
        params=params,
        s_grid=s_grid,
        v0=float(v0),
        strike=MC_STRIKE,
        r=MC_RATE,
        maturity=MC_MATURITY,
        n_paths=MC_PATHS,
        n_steps=MC_STEPS,
        seed=MC_SEED + j,
        device=mc_device,
    )

    mc_price_columns.append(result.price)
    mc_se_columns.append(result.standard_error)
    mc_low_columns.append(result.ci95_low)
    mc_high_columns.append(result.ci95_high)

    if (j + 1) % 4 == 0 or j == 0 or j == len(v_grid) - 1:
        print(f'Completed variance slice {j + 1}/{len(v_grid)}')

mc_surface_time = time.perf_counter() - mc_surface_start

# 32 vectors of length 64 -> one [64, 32] surface.
mc_surface = torch.stack(mc_price_columns, dim=1)
mc_se_surface = torch.stack(mc_se_columns, dim=1)
mc_low_surface = torch.stack(mc_low_columns, dim=1)
mc_high_surface = torch.stack(mc_high_columns, dim=1)

assert tuple(fno_surface.shape) == (64, 32)
assert tuple(adi_surface.shape) == (64, 32)
assert tuple(mc_surface.shape) == (64, 32)

print('FNO surface shape:', tuple(fno_surface.shape))
print('ADI surface shape:', tuple(adi_surface.shape))
print('MC surface shape :', tuple(mc_surface.shape))
print(f'Monte Carlo full-surface time: {mc_surface_time:.6f} s')


In [ ]:
# ============================================================
# 18C. MONTE CARLO SURFACE AND ERROR SURFACES
# ============================================================

# Monte Carlo pricing surface
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(S_np, V_np, mc_surface.numpy(), cmap='viridis', linewidth=0)
ax.set_xlabel('Spot price S')
ax.set_ylabel('Variance v')
ax.set_zlabel('Option price')
ax.set_title('Monte Carlo — Complete 64 × 32 Pricing Surface')
fig.colorbar(surf, ax=ax, shrink=0.65, label='Option price')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'monte_carlo_complete_surface.png', dpi=200)
plt.show()

# FNO vs Monte Carlo absolute error
fno_mc_error = torch.abs(fno_surface - mc_surface)
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(S_np, V_np, fno_mc_error.numpy(), cmap='magma', linewidth=0)
ax.set_xlabel('Spot price S')
ax.set_ylabel('Variance v')
ax.set_zlabel('Absolute error')
ax.set_title('FNO vs Monte Carlo — Absolute Error')
fig.colorbar(surf, ax=ax, shrink=0.65, label='Absolute error')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fno_vs_monte_carlo_error_surface.png', dpi=200)
plt.show()

# ADI vs Monte Carlo absolute error
adi_mc_error = torch.abs(adi_surface - mc_surface)
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(S_np, V_np, adi_mc_error.numpy(), cmap='magma', linewidth=0)
ax.set_xlabel('Spot price S')
ax.set_ylabel('Variance v')
ax.set_zlabel('Absolute error')
ax.set_title('ADI vs Monte Carlo — Absolute Error')
fig.colorbar(surf, ax=ax, shrink=0.65, label='Absolute error')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'adi_vs_monte_carlo_error_surface.png', dpi=200)
plt.show()

# Monte Carlo confidence-interval width
mc_ci_width = mc_high_surface - mc_low_surface
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(S_np, V_np, mc_ci_width.numpy(), cmap='plasma', linewidth=0)
ax.set_xlabel('Spot price S')
ax.set_ylabel('Variance v')
ax.set_zlabel('95% CI width')
ax.set_title('Monte Carlo — 95% Confidence-Interval Width')
fig.colorbar(surf, ax=ax, shrink=0.65, label='95% CI width')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'monte_carlo_ci_width_surface.png', dpi=200)
plt.show()


## 19. FNO vs ADI vs Monte Carlo — complete-surface comparison

All numerical comparisons below use the same selected test parameter vector and the same 64 × 32 spatial grid.


In [ ]:
# ============================================================
# 19A. WHOLE-SURFACE METRICS
# ============================================================

def surface_comparison_metrics(prediction, reference):
    prediction = prediction.float()
    reference = reference.float()
    diff = prediction - reference

    return {
        'MAE': float(torch.mean(torch.abs(diff)).item()),
        'RMSE': float(torch.sqrt(torch.mean(diff ** 2)).item()),
        'Relative L2': float(
            (
                torch.linalg.norm(diff.reshape(-1), ord=2)
                / (torch.linalg.norm(reference.reshape(-1), ord=2) + 1e-7)
            ).item()
        ),
        'Maximum Absolute Error': float(torch.max(torch.abs(diff)).item()),
    }

fno_vs_adi_full = surface_comparison_metrics(fno_surface, adi_surface)
fno_vs_mc_full = surface_comparison_metrics(fno_surface, mc_surface)
adi_vs_mc_full = surface_comparison_metrics(adi_surface, mc_surface)

comparison_df = pd.DataFrame([
    {'Comparison': 'FNO vs ADI', **fno_vs_adi_full},
    {'Comparison': 'FNO vs Monte Carlo', **fno_vs_mc_full},
    {'Comparison': 'ADI vs Monte Carlo', **adi_vs_mc_full},
])

display(comparison_df)
comparison_df.to_csv(RESULTS_DIR / 'fno_adi_monte_carlo_surface_comparison.csv', index=False)


In [ ]:
# ============================================================
# 19B. 1D CROSS-SECTION THROUGH THE THREE COMPLETE SURFACES
# ============================================================

v_index = len(v_grid) // 2
selected_v = float(v_grid[v_index])

plt.figure(figsize=(10, 6))
plt.plot(s_grid.numpy(), fno_surface[:, v_index].numpy(), label='FNO', linewidth=2)
plt.plot(s_grid.numpy(), adi_surface[:, v_index].numpy(), label='ADI', linewidth=2)
plt.plot(s_grid.numpy(), mc_surface[:, v_index].numpy(), label='Monte Carlo', linewidth=2)
plt.fill_between(
    s_grid.numpy(),
    mc_low_surface[:, v_index].numpy(),
    mc_high_surface[:, v_index].numpy(),
    alpha=0.2,
    label='MC 95% CI',
)
plt.xlabel('Spot price S')
plt.ylabel('European call price')
plt.title(f'FNO vs ADI vs Monte Carlo at v = {selected_v:.6f}')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fno_adi_monte_carlo_cross_section.png', dpi=200)
plt.show()


In [ ]:
# ============================================================
# 19C. MONTE CARLO 95% CI COVERAGE
# ============================================================

fno_inside_mc_ci = ((fno_surface >= mc_low_surface) & (fno_surface <= mc_high_surface)).float().mean().item()
adi_inside_mc_ci = ((adi_surface >= mc_low_surface) & (adi_surface <= mc_high_surface)).float().mean().item()

ci_coverage_df = pd.DataFrame([
    {'Method': 'FNO', 'Fraction inside MC 95% CI': fno_inside_mc_ci},
    {'Method': 'ADI', 'Fraction inside MC 95% CI': adi_inside_mc_ci},
])

display(ci_coverage_df)
ci_coverage_df.to_csv(RESULTS_DIR / 'monte_carlo_ci_coverage.csv', index=False)


## 20. Timing: complete 64 × 32 surface

The timings below measure the wall-clock time needed to generate the complete surface for the same selected test sample and the same 64 × 32 grid. The table records the device and workload so the hardware difference is explicit.


In [ ]:
# ============================================================
# 20A. FNO COMPLETE-SURFACE TIMING
# ============================================================

FNO_SURFACE_REPEATS = 5
fno_surface_times = []

with torch.no_grad():
    _ = model(x_scaled.to(device))
if device.type == 'cuda':
    torch.cuda.synchronize()

for _ in range(FNO_SURFACE_REPEATS):
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(x_scaled.to(device))
    if device.type == 'cuda':
        torch.cuda.synchronize()
    fno_surface_times.append(time.perf_counter() - start)

fno_surface_avg_time = float(np.mean(fno_surface_times))
print(f'FNO complete-surface average time: {fno_surface_avg_time:.6f} s')


In [ ]:
# ============================================================
# 20B. ADI COMPLETE-SURFACE TIMING
# ============================================================

ADI_SURFACE_REPEATS = 3
adi_surface_times = []

_ = solve_heston_adi(
    params=params,
    s_grid=s_grid,
    v_grid=v_grid,
    strike=MC_STRIKE,
    r=MC_RATE,
    maturity=MC_MATURITY,
    n_steps=ADI_N_STEPS,
)

for _ in range(ADI_SURFACE_REPEATS):
    start = time.perf_counter()
    _ = solve_heston_adi(
        params=params,
        s_grid=s_grid,
        v_grid=v_grid,
        strike=MC_STRIKE,
        r=MC_RATE,
        maturity=MC_MATURITY,
        n_steps=ADI_N_STEPS,
    )
    adi_surface_times.append(time.perf_counter() - start)

adi_surface_avg_time = float(np.mean(adi_surface_times))
print(f'ADI complete-surface average time: {adi_surface_avg_time:.6f} s')


In [ ]:
# ============================================================
# 20C. FINAL TIMING TABLE AND SPEED RATIOS
# ============================================================

surface_timing_df = pd.DataFrame([
    {
        'Method': 'FNO',
        'Surface': '64 × 32',
        'Time (s)': fno_surface_avg_time,
        'Device': str(device),
        'Workload': '1 FNO forward pass',
    },
    {
        'Method': 'ADI',
        'Surface': '64 × 32',
        'Time (s)': adi_surface_avg_time,
        'Device': 'CPU (NumPy)',
        'Workload': f'{ADI_N_STEPS} ADI time steps',
    },
    {
        'Method': 'Monte Carlo',
        'Surface': '64 × 32',
        'Time (s)': mc_surface_time,
        'Device': str(mc_device),
        'Workload': f'{MC_PATHS} paths × {MC_STEPS} steps × 32 variance slices',
    },
])

display(surface_timing_df)
surface_timing_df.to_csv(RESULTS_DIR / 'fno_adi_monte_carlo_surface_timing.csv', index=False)

print(f"ADI / FNO surface-time ratio: {adi_surface_avg_time / max(fno_surface_avg_time, 1e-12):.2f}x")
print(f"MC / FNO surface-time ratio: {mc_surface_time / max(fno_surface_avg_time, 1e-12):.2f}x")

plt.figure(figsize=(8, 5))
plt.bar(surface_timing_df['Method'], surface_timing_df['Time (s)'])
plt.ylabel('Time (seconds)')
plt.title('Time to Generate Complete 64 × 32 Pricing Surface')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fno_adi_monte_carlo_surface_timing.png', dpi=200)
plt.show()


### Interpretation note

The dataset reference in this project was generated using ADI, so FNO-versus-ADI measures agreement with the PDE reference used for training. Monte Carlo provides an additional numerical consistency check, but Monte Carlo is itself approximate: its results include sampling uncertainty and Euler time-discretisation error. The timing table uses the same test parameter set and spatial grid, while explicitly reporting each method's device and workload.


## 21. Save a compact report summary


In [ ]:
report_summary = {
    "device": str(device),
    "checkpoint": str(CHECKPOINT_PATH),
    "test_samples_evaluated": len(test_scaled_dataset),
    "surface_test_position": SURFACE_TEST_POSITION,
    "surface_original_dataset_index": surface_raw_index,
    "RMSE": rmse_value,
    "MAE": mae_value,
    "Maximum Absolute Error": max_abs,
    "Mean Relative Error": mre_value,
    "Relative L2 Error": relative_l2_value,
    "R2": r2_value,
    "FNO average inference time (s/batch)": fno_time,
    "FNO throughput (samples/s)": fno_throughput,
    "ADI complete-surface average time (s)": adi_surface_avg_time,
    "Monte Carlo full-surface time (s)": mc_surface_time,
    "Monte Carlo paths per variance slice": MC_PATHS,
    "Monte Carlo Euler steps": MC_STEPS,
    "MC device": str(mc_device),
}


summary_df = pd.DataFrame(
    [{"Metric": k, "Value": v} for k, v in report_summary.items()]
)
display(summary_df)

with open(RESULTS_DIR / "results_summary.txt", "w", encoding="utf-8") as f:
    for key, value in report_summary.items():
        f.write(f"{key}: {value}\n")

print("\nAll result files saved under:")
print(RESULTS_DIR)